In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
import torch
import time
import numpy as np
from vllm import LLM, SamplingParams
from vllm.v1.metrics.reader import Counter, Gauge, Histogram, Vector

WARNING 02-09 14:51:04 [cuda.py:608] Detected different devices in the system: Tesla V100-PCIE-32GB, Tesla V100S-PCIE-32GB, Tesla V100S-PCIE-32GB, Tesla V100-PCIE-32GB. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.


In [ ]:
# model defintion
model_id = "../Llama-3.2-1B"
llm = LLM(model_id, disable_log_stats=False)

# model prompts
prompts = [
    "Hello, my name is",
    "San Francisco is a",
    "The capital of France is",
    "The future of AI is",
]
sampling_params = SamplingParams(temperature=0.4, top_p=0.95, max_tokens=128)

INFO 02-09 14:51:19 [utils.py:253] non-default args: {'model': './Llama-3.2-1B'}
INFO 02-09 14:51:19 [model.py:631] Resolved architecture: LlamaForCausalLM
WARNING 02-09 14:51:19 [model.py:1921] Your device 'Tesla V100-PCIE-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 02-09 14:51:19 [model.py:1971] Casting torch.bfloat16 to torch.float16.
INFO 02-09 14:51:19 [model.py:1745] Using max model len 131072
INFO 02-09 14:51:21 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:22 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='./Llama-3.2-1B', speculative_config=None, tokenizer='./Llama-3.2-1B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_p

(EngineCore_DP0 pid=2249376) /home/kpenners/.venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=2249376) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=2249376)   warnings.warn(


(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:25 [cuda.py:418] Valid backends: ['TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:25 [cuda.py:427] Using TRITON_ATTN backend.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:26 [default_loader.py:314] Loading weights took 0.75 seconds
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:27 [gpu_model_runner.py:3338] Model loading took 2.3185 GiB memory and 2.935443 seconds
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:31 [backends.py:631] Using cache directory: /home/kpenners/.cache/vllm/torch_compile_cache/7259afe49c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:31 [backends.py:647] Dynamo bytecode transform time: 3.69 s
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:32 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.063 s
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:33 [monitor.py:34] torch.compile takes 4.75 s in total
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:34 [gpu_worker.py:359] Available KV cache memory: 25.01 GiB
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:34 [kv_cache_utils.py:1229] GPU KV cache size: 819,392 toke

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 33.40it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 34.52it/s]


(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:37 [gpu_model_runner.py:4244] Graph capturing finished in 3 secs, took 0.29 GiB
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:37 [core.py:250] init engine (profile, create kv cache, warmup model) took 10.50 seconds
INFO 02-09 14:51:37 [gpu_model_runner.py:4244] Graph capturing finished in 3 secs, took 0.29 GiB
(EngineCore_DP0 pid=2249376) INFO 02-09 14:51:37 [core.py:250] init engine (profile, create kv cache, warmup model) took 10.50 seconds
INFO 02-09 14:51:38 [llm.py:352] Supported tasks: ['generate']


In [ ]:
# warm up
_ = llm.generate(prompts)

WARNING 02-09 14:51:42 [model.py:1568] Default sampling parameters have been overridden by the model's Hugging Face generation config recommended from the model creator. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.


Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
outputs = llm.generate(prompts, sampling_params)

# print outpus
print("-" * 40)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"prompt: {prompt!r}\n generated text: {generated_text!r}")
    print(f"metrics: \n{output.metrics}")
    print("-" * 40)


for metric in llm.get_metrics():
    if isinstance(metric, Gauge):
            print(f"{metric.name} (gauge) = {metric.value}")
    elif isinstance(metric, Counter):
        print(f"{metric.name} (counter) = {metric.value}")
    elif isinstance(metric, Vector):
        print(f"{metric.name} (vector) = {metric.values}")
    elif isinstance(metric, Histogram):
        print(f"{metric.name} (histogram)")
        print(f"    sum = {metric.sum}")
        print(f"    count = {metric.count}")
        for bucket_le, value in metric.buckets.items():
            print(f"    {bucket_le} = {value}")

Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

----------------------------------------
prompt: 'Hello, my name is'
 generated text: ' Chris and I am a 19 year old student from the UK. I am currently studying for a BSc in Psychology at the University of Nottingham. I am interested in the study of human behaviour and am currently undertaking a research project into the effects of music on mood and behaviour. I am also interested in the application of psychology in the workplace and am currently undertaking an internship at a local company. My interests include music, video games, psychology and the outdoors.'
metrics: 
RequestStateStats(num_generation_tokens=93, arrival_time=1770645110.69595, queued_ts=7011694.982319483, scheduled_ts=7011694.982405514, first_token_ts=7011694.990249356, last_token_ts=7011695.563146179, first_token_latency=0.009682416915893555, is_corrupted=False)
----------------------------------------
prompt: 'San Francisco is a'
 generated text: ' city of contrasts. It has the tallest building in the Western Hemis

In [ ]:
for prompt in prompts:
    start_time = time.perf_counter()
    outputs = llm.generate(prompt, sampling_params)
    end_time = time.perf_counter()

    output = outputs[0].outputs[0]
    text = output.text

    print(f"prompt: {prompt!r}")
    print(f"generated text: {text!r}")
    print(f"elapsed time: {end_time - start_time:.2f} s")
    print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print("-" * 40 )

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

INFO 02-09 14:53:09 [loggers.py:236] Engine 000: Avg prompt throughput: 0.5 tokens/s, Avg generation throughput: 5.9 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
prompt: 'Hello, my name is'
generated text: ' Anna and I am a 3rd year PhD student in the Department of Psychology at the University of Sheffield. My research focuses on the neural mechanisms underlying the perception and processing of emotional facial expressions. I am interested in the neural correlates of emotional processing, and how these relate to individual differences in emotional perception.\nI am also a member of the Sheffield Emotion Lab, which is a group of researchers from the Department of Psychology and the Department of Computer Science who are interested in the study of emotion. We are currently investigating the neural mechanisms underlying the perception of emotional facial expressions, and how these relate to individual differences in emotional perception

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

prompt: 'The capital of France is'
generated text: ' Paris. It is the most visited city in the world, with 12 million visitors a year. It is also the most expensive city in the world, with an average cost of living of €1,600 per month. Paris is a city of contrasts, with its ancient monuments and modern skyscrapers, its narrow streets and wide boulevards, its chic boutiques and its red-light district. It is a city of art, culture and history, with a rich and diverse heritage. It is also a city of fashion, with its chic boutiques and its famous fashion designers.\nParis is a city of contrasts, with its ancient'
elapsed time: 0.69 s
GPU memory used: 0.00 GB
----------------------------------------


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

prompt: 'The future of AI is'
generated text: ' in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe future of AI is in the hands of the people\nThe'
elapsed time: 0.69 s
GPU memory used: 0.00 GB
----------------------------------------
